In [ ]:
import os

os.chdir("..")

In [ ]:
# from sklearn.preprocessing import SplineTransformer
# from sklearn.linear_model import Ridge
# from patsy import dmatrix
import bambi as bmb
from prprod_repl import ProdRepl
import polars as pl
import arviz_plots as azp
import arviz as az

azp.style.use("arviz-variat")


pr = ProdRepl()

In [ ]:
df = pr.reg_data()
df = df.with_columns(
    capital_synth_log=pl.col("capital_synth").log(),
    labour_log=pl.col("labour").log(),
    energy_log=pl.col("energy"),
).sort(["year", "month", "muni"])
df = df.to_pandas()
df

In [ ]:
model = bmb.Model(
    "energy_log ~ capital_synth_log + labour_log",
    data=df,
    family="gaussian"
)
idata = model.fit(
    draws=1000,
    tune=1000,
    target_accept=0.9
)

In [ ]:
import pymc as pm
import numpy as np
import pandas as pd

# 1. Preprocessing
# Ensure 'muni' is categorical to get integer indices for the hierarchy
df['muni_idx'], muni_labels = pd.factorize(df['muni'])
n_munis = len(muni_labels)
coords = {"muni": muni_labels, "obs": np.arange(len(df))}

with pm.Model(coords=coords) as cobb_douglas_model:
    # Data containers
    muni_idx = pm.Data("muni_idx", df['muni_idx'], dims="obs")
    log_k = pm.Data("log_k", df['capital_synth_log'], dims="obs")
    log_l = pm.Data("log_l", df['labour_log'], dims="obs")
    log_y = pm.Data("log_y", df['energy_log'], dims="obs") # Replace with your DV name

    # --- Hyperpriors (Global means and standard deviations) ---
    mu_alpha = pm.Normal("mu_alpha", mu=0, sigma=5)
    sigma_alpha = pm.Exponential("sigma_alpha", 1.0)
    
    mu_beta_k = pm.Normal("mu_beta_k", mu=0.3, sigma=1) # Capital share usually ~0.3
    sigma_beta_k = pm.Exponential("sigma_beta_k", 1.0)
    
    mu_beta_l = pm.Normal("mu_beta_l", mu=0.7, sigma=1) # Labour share usually ~0.7
    sigma_beta_l = pm.Exponential("sigma_beta_l", 1.0)

    # --- Varying Coefficients (Non-Centered Parameterization for better sampling) ---
    z_alpha = pm.Normal("z_alpha", mu=0, sigma=1, dims="muni")
    alpha = pm.Deterministic("alpha", mu_alpha + z_alpha * sigma_alpha, dims="muni")
    
    z_beta_k = pm.Normal("z_beta_k", mu=0, sigma=1, dims="muni")
    beta_k = pm.Deterministic("beta_k", mu_beta_k + z_beta_k * sigma_beta_k, dims="muni")
    
    z_beta_l = pm.Normal("z_beta_l", mu=0, sigma=1, dims="muni")
    beta_l = pm.Deterministic("beta_l", mu_beta_l + z_beta_l * sigma_beta_l, dims="muni")

    # --- Likelihood ---
    # Linear model: log(Y) = alpha[muni] + beta_k[muni]*log(K) + beta_l[muni]*log(L)
    mu = alpha[muni_idx] + beta_k[muni_idx] * log_k + beta_l[muni_idx] * log_l
    
    sigma_eps = pm.Exponential("sigma_eps", 1.0)
    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma_eps, observed=log_y, dims="obs")

    # --- Inference ---
    idata = pm.sample(2000, tune=1000, target_accept=0.9)

In [ ]:
import pymc as pm
import numpy as np
import pandas as pd

# 1. Preprocessing Time
# Create a continuous time index (e.g., 0, 1, 2...) to represent months
df['time_idx'] = (df['year'] - df['year'].min()) * 12 + (df['month'] - 1)
time_vec = df['time_idx'].values[:, None] # GP needs 2D input (N, 1)

df['muni_idx'], muni_labels = pd.factorize(df['muni'])
n_munis = len(muni_labels)
coords = {"muni": muni_labels, "obs": np.arange(len(df))}

with pm.Model(coords=coords) as cobb_douglas_gp_model:
    # Data containers
    muni_idx = pm.Data("muni_idx", df['muni_idx'], dims="obs")
    log_k = pm.Data("log_k", df['capital_synth_log'], dims="obs")
    log_l = pm.Data("log_l", df['labour_log'], dims="obs")
    log_y = pm.Data("log_y", df['energy_log'], dims="obs")
    t = pm.Data("t", time_vec, dims=("obs", "feature"))

    # --- Hyperpriors ---
    mu_alpha = pm.Normal("mu_alpha", mu=0, sigma=5)
    sigma_alpha = pm.HalfNormal("sigma_alpha", 1.0)
    
    mu_beta_k = pm.Normal("mu_beta_k", mu=0.3, sigma=0.1)
    sigma_beta_k = pm.HalfNormal("sigma_beta_k", 0.5)
    
    mu_beta_l = pm.Normal("mu_beta_l", mu=0.7, sigma=0.1)
    sigma_beta_l = pm.HalfNormal("sigma_beta_l", 0.5)

    # --- Varying Coefficients (Muni-level) ---
    z_alpha = pm.Normal("z_alpha", 0, 1, dims="muni")
    alpha = pm.Deterministic("alpha", mu_alpha + z_alpha * sigma_alpha, dims="muni")
    
    z_beta_k = pm.Normal("z_beta_k", 0, 1, dims="muni")
    beta_k = pm.Deterministic("beta_k", mu_beta_k + z_beta_k * sigma_beta_k, dims="muni")
    
    z_beta_l = pm.Normal("z_beta_l", 0, 1, dims="muni")
    beta_l = pm.Deterministic("beta_l", mu_beta_l + z_beta_l * sigma_beta_l, dims="muni")

    # --- Gaussian Process for Time Trend ---
    # Lengthscale: How quickly the 'productivity' changes over months
    # Amplitude: How much the productivity fluctuates
    ls = pm.InverseGamma("ls", mu=12, sigma=3) # Prior: changes happen roughly yearly
    eta = pm.Exponential("eta", 1.0)
    cov_func = eta**2 * pm.gp.cov.ExpQuad(1, ls=ls)
    
    # HSGP is much faster for N > 500
    gp = pm.gp.HSGP(m=[20], L=[time_vec.max() * 1.2], cov_func=cov_func)
    phi_t = gp.prior("phi_t", X=t, dims="obs")

    # --- Likelihood ---
    # The GP (phi_t) acts as a time-varying Total Factor Productivity (TFP)
    mu = (alpha[muni_idx] + 
          beta_k[muni_idx] * log_k + 
          beta_l[muni_idx] * log_l + 
          phi_t)
    
    sigma_eps = pm.Exponential("sigma_eps", 1.0)
    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma_eps, observed=log_y, dims="obs")

    # --- Inference ---
    idata = pm.sample(1000, tune=1000, target_accept=0.95)

In [ ]:
az.summary(idata)

In [ ]:
import arviz as az

# 1. Get Summary for coefficients
summary = az.summary(idata, var_names=["beta_k", "beta_l"])

# 2. Extract GP Trend (phi_t)
# We take the mean across the posterior samples for each time point
post_phi = idata.posterior["phi_t"].mean(dim=["chain", "draw"])

# 3. Forest Plot (Quick check for all munis)
az.plot_forest(idata, var_names=["beta_k", "beta_l"], combined=True)

In [ ]:
import arviz as az

# Extract only the global (state-level) means
state_results = az.summary(idata, var_names=["mu_alpha", "mu_beta_k", "mu_beta_l"])

print("--- State-Level Production Coefficients ---")
print(state_results)

# To get the specific mean values as a dictionary
means = state_results['mean'].to_dict()
print(f"\nState Capital Elasticity: {means['mu_beta_k']:.3f}")
print(f"State Labour Elasticity: {means['mu_beta_l']:.3f}")

In [ ]:
import matplotlib.pyplot as plt

# Plot the posterior distributions for the state-level elasticities
az.plot_posterior(idata, var_names=["mu_beta_k", "mu_beta_l"], 
                  ref_val=[0.3, 0.7], # Comparing against standard economic benchmarks
                  color='skyblue')
plt.suptitle("State-Level Elasticity Posteriors")
plt.show()

In [ ]:
import pymc as pm
import arviz as az

sigmas_to_test = [0.1, 0.5, 10.0]
results = {}

for s in sigmas_to_test:
    with pm.Model() as test_model:
        # Standardize your inputs for better comparison
        mu_beta_k = pm.Normal("mu_beta_k", mu=0.3, sigma=s)
        mu_beta_l = pm.Normal("mu_beta_l", mu=0.7, sigma=s)
        
        # ... (rest of your model code here) ...
        
        trace = pm.sample(1000, tune=1000, target_accept=0.9, progressbar=False)
        results[f"sigma_{s}"] = trace

# Compare the posteriors visually
az.plot_density(
    [results["sigma_0.1"], results["sigma_0.5"], results["sigma_10.0"]],
    var_names=["mu_beta_k", "mu_beta_l"],
    data_labels=["Strong (0.1)", "Moderate (0.5)", "Flat (10.0)"]
)

In [ ]:
import pandas as pd
import arviz as az

# List to store summary dataframes
summary_list = []

for label, trace in results.items():
    # Calculate summary statistics for the hyperpriors
    summary = az.summary(trace, var_names=["mu_beta_k", "mu_beta_l"], round_to=3)
    
    # Add a column to identify which prior sigma this belongs to
    summary["prior_sigma"] = label
    summary_list.append(summary)

# Combine all summaries into one table
final_comparison = pd.concat(summary_list)

# Reorder columns for readability
cols = ['prior_sigma', 'mean', 'sd', 'hdi_3%', 'hdi_97%', 'ess_bulk', 'r_hat']
final_comparison = final_comparison[cols]

print("--- Prior Sensitivity Numerical Comparison ---")
print(final_comparison)

# Optional: specifically print the shift in Returns to Scale
print("\n--- Returns to Scale (RTS) Drift ---")
for label, trace in results.items():
    k_mean = trace.posterior["mu_beta_k"].mean().values
    l_mean = trace.posterior["mu_beta_l"].mean().values
    print(f"{label}: RTS = {k_mean + l_mean:.3f}")

In [ ]:
import pandas as pd
import arviz as az

# 1. Extract the varying coefficients for all municipalities
muni_betas = az.summary(idata, var_names=["beta_k", "beta_l"], round_to=3)

# 2. Add the county names back to the summary
# muni_labels was the list of names from pd.factorize earlier
muni_betas['county'] = list(muni_labels) * 2 # One for beta_k, one for beta_l
muni_betas['parameter'] = (['Capital'] * n_munis) + (['Labour'] * n_munis)

# 3. Pivot for easy reading
pivot_betas = muni_betas.pivot(index='county', columns='parameter', values='mean')
pivot_betas['Total_RTS'] = pivot_betas['Capital'] + pivot_betas['Labour']

# Sort by the most capital-efficient counties
print(pivot_betas.sort_values(by='Capital', ascending=False).head(10))